
# Vision RAG using Groq + Gradio

This notebook builds a simple Vision-RAG style pipeline:
- Accepts an image
- Extracts visual description
- Uses Groq LLM to generate a **~10 word explanation**
- Deploys via Gradio

You can enter your **Groq API Key** during runtime.


In [ ]:

!pip install groq gradio pillow


In [ ]:

from groq import Groq
from PIL import Image
import gradio as gr
import base64
import io


In [ ]:

def encode_image(image):
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


In [ ]:

def vision_rag(image, api_key):
    if image is None:
        return "Please upload an image."

    client = Groq(api_key=api_key)

    img_base64 = encode_image(image)

    prompt = "Describe this image in about 10 words only."

    response = client.chat.completions.create(
        model="llama-3.2-90b-vision-preview",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": f"data:image/png;base64,{img_base64}",
                    },
                ],
            }
        ],
    )

    return response.choices[0].message.content


In [ ]:

def app():
    with gr.Blocks() as demo:
        gr.Markdown("## Vision RAG (Groq) - 10 Word Image Explainer")

        api_key = gr.Textbox(label="Enter Groq API Key", type="password")
        image = gr.Image(type="pil")
        output = gr.Textbox(label="Explanation")

        btn = gr.Button("Generate")

        btn.click(vision_rag, inputs=[image, api_key], outputs=output)

    return demo

demo = app()
demo.launch()
